# 04 · Evaluation
Cell-level metrics (smoke F1, malignancy AUC), subject-level metrics (cancer ROC-AUC, sensitivity/specificity at the clinical HIGH RISK threshold), and attention interpretability — one shared subject forward pass across all three, via `Evaluator.full_report()`.

In [ ]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
from evaluate import Evaluator
from train import CellLevelDataset, SubjectLevelDataset

CFG = '../configs/default.yaml'
PHASE = 3   # load checkpoints/phase3_best.pt

## Load trained model + preprocessed data
Falls back to an untrained model on synthetic data if no checkpoint or processed dataset exists yet — run [03_training.ipynb](03_training.ipynb) first for meaningful metrics.

In [ ]:
import numpy as np, random
from constants import N_SMOKE_CLASSES, N_CELL_TYPES

try:
    ev = Evaluator.from_checkpoint(CFG, phase=PHASE)
except FileNotFoundError:
    print('No checkpoint found — run 03_training.ipynb first. Using untrained model for demo.')
    from model import MultiSmokeCancerNet
    ev = Evaluator(MultiSmokeCancerNet.from_config(CFG))

processed = Path('../data/processed')
if processed.exists() and (processed / 'gene_matrix.npy').exists():
    cell_ds = CellLevelDataset.from_dir(processed)
else:
    print('No real data — run 02_preprocessing.ipynb first. Using synthetic data for demo.')
    cell_ds = CellLevelDataset(
        gene_matrix=np.random.randn(500, 2000).astype('float32'),
        smoke_labels=np.random.randint(0, N_SMOKE_CLASSES, 500),
        malignancy_labels=np.random.randint(0, 2, 500).astype('float32'),
        cell_type_ids=np.random.randint(0, N_CELL_TYPES, 500),
    )

def _bag(n):
    return {'gene_matrix': np.random.randn(n, 2000).astype('float32'),
            'cell_type_ids': np.random.randint(0, N_CELL_TYPES, n),
            'smoke_labels': np.random.randint(0, N_SMOKE_CLASSES, n),
            'malig_labels': np.random.randint(0, 2, n).astype('float32'),
            'cancer_label': random.randint(0, 1)}

subject_ds = SubjectLevelDataset([
    {'subject_id': f's{i}', **_bag(random.randint(40, 100))} for i in range(20)
])

## Full report
Single subject forward pass shared across subject-level metrics and interpretability — saved to `checkpoints/evaluation_report.json`.

In [ ]:
report, raw = ev.full_report(cell_ds, subject_ds)
report

## Plots
ROC/PR curves, confusion matrix, calibration, attention-by-cell-type, attention-by-smoke-type.

In [ ]:
from pathlib import Path
from visualize import plot_all
from IPython.display import Image

plot_all(
    report,
    y_true_cancer=raw['y_true_cancer'],
    y_prob_cancer=raw['y_prob_cancer'],
    history_dir='../checkpoints',
    out_dir='../checkpoints/plots',
)

roc_png = Path('../checkpoints/plots/roc_curve.png')
Image(str(roc_png)) if roc_png.exists() else print('ROC needs both classes present in subject_ds')